In [1]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [3]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data/hw/sionna/ws/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data/hw/sionna/ws/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm__38         

In [4]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data/hw/sionna/ws/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data/hw/sionna/ws/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [5]:
scene.preview(point_picker=True)

In [6]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [7]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [8]:
scene.preview(point_picker=True)

bs 1
ue 7
V 

In [ ]:
import os
import tensorflow as tf
import numpy as np
import sionna

# [버전 호환성] Import 경로 처리
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver
try:
    from sionna.phy.ofdm import ResourceGrid
except ImportError:
    from sionna.ofdm import ResourceGrid

# ==========================================
# 1. 파일 저장 경로 설정 (요청 사항 반영)
# ==========================================
# 저장할 디렉토리 경로
output_dir = "/data/hw/sionna/ws/output"

# 디렉토리가 없으면 생성
if not os.path.exists(output_dir):
    try:
        os.makedirs(output_dir)
        print(f"디렉토리 생성됨: {output_dir}")
    except OSError as e:
        print(f"[오류] 디렉토리를 생성할 수 없습니다: {e}")
        # 실패 시 현재 디렉토리에 저장하도록 fallback
        output_dir = "."

# Scene 로딩을 위한 경로 설정 (기존 유지)
xml_path = "/data/hw/sionna/ws/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# ==========================================
# 2. 시스템 및 시뮬레이션 파라미터 (정밀 설정)
# ==========================================
carrier_frequency = 3.5e9
subcarrier_spacing = 30e3 
fft_size = 72
num_ofdm_symbols = 14

# [수정됨] 정밀 시뮬레이션 설정
simulation_time = 10.0 # 10초
dt = 0.5e-3 # 0.5 ms (30kHz SCS의 1 Slot 길이)
total_steps = int(simulation_time / dt)

print(f"--- 시뮬레이션 설정 ---")
print(f"총 시간: {simulation_time}초")
print(f"시간 간격(dt): {dt}초 (0.5ms)")
print(f"총 스텝 수: {total_steps} (메모리 주의)")

# ==========================================
# 3. 경로 및 기지국 설정
# ==========================================
# 기지국(BS) 위치
bs_position = [323.47, 36.87, -204.31]

# 이동성 설정
num_ues = 8
speeds = [10, 15, 20, 25, 30, 40, 50, 60]

# [필수] 이전 셀에서 생성된 road_positions 확인
if 'road_positions' not in globals() or len(road_positions) == 0:
    raise ValueError("메모리에 'road_positions' 변수가 없습니다. 도로 좌표 추출 코드를 먼저 실행해주세요.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
trajectory_points = road_positions[path_indices]

# PolylineWalker 클래스 정의
class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)
        self.total_length = self.cum_dist[-1]
        
    def get_position(self, distance):
        if distance >= self.total_length: return self.points[-1]
        if distance <= 0: return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        idx = max(0, idx)
        p_start = self.points[idx]
        p_end = self.points[idx+1]
        seg_len = self.seg_lengths[idx]
        ratio = (distance - self.cum_dist[idx]) / seg_len if seg_len > 0 else 0
        return p_start + (p_end - p_start) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 4. Scene 구성
# ==========================================
scene = load_scene(temp_xml_path)

# 안테나: BS(8x8), UE(1x1)
bs_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")
ue_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

scene.tx_array = bs_array
scene.rx_array = ue_array

# 기기 배치
tx = Transmitter(name="BS", position=bs_position, orientation=[0,0,0])
scene.add(tx)

ues = []
start_pos = trajectory_points[0]
for i in range(num_ues):
    rx = Receiver(name=f"UE_{i}", position=start_pos, orientation=[0,0,0])
    scene.add(rx)
    ues.append(rx)

# ==========================================
# 5. 시뮬레이션 루프 (대용량 데이터 생성)
# ==========================================
solver = PathSolver()
dataset_h = [] # 여기에 20,000개의 데이터가 쌓입니다.

print("시뮬레이션 시작...")

for step in range(total_steps):
    current_time = step * dt
    
    # 이동
    for i, ue in enumerate(ues):
        speed = speeds[i] / 3.6
        new_pos = walker.get_position(speed * current_time)
        ue.position = new_pos
    
    # Ray Tracing
    paths = solver(scene, max_depth=3)
    
    # CFR 계산
    frequencies = subcarrier_spacing * tf.range(fft_size, dtype=tf.float32)
    cfr_output = paths.cfr(frequencies=frequencies)

    real = cfr_output[0]
    imag = cfr_output[1]

    #print(len(cfr_output))
    #print(real.shape)
    #print(imag.shape)
    #print("real: ", real[0])
    #print("imag: ", imag[1])
    #print("cfr: ", cfr_output)
    
    # numpy 변환
    real = real.numpy()
    imag = imag.numpy()

    # squeeze → (8,64,72)
    real = np.squeeze(real)
    imag = np.squeeze(imag)

    # 마지막 축에 real/imag 쌓기
    H = np.stack([real, imag], axis=-1)   # (8,64,72,2)

    dataset_h.append(H)
    
    # 진행 상황 출력 (너무 자주 출력하지 않도록 1000스텝마다)
    if step % 1000 == 0:
        print(f"Progress: {step}/{total_steps} ({(step/total_steps)*100:.1f}%)")

# ==========================================
# 6. 저장
# ==========================================
print("데이터 변환 중... (시간이 소요될 수 있습니다)")
dataset_h = np.array(dataset_h)
dataset_h = np.squeeze(dataset_h)

print(f"최종 데이터 형태: {dataset_h.shape}")
# 예상: (20000, 8, 64, 72)

file_name = "bs1_ue7_channel_dataset_precise_10s.npy"
save_path = os.path.join(output_dir, file_name)

np.save(save_path, dataset_h)
print(f"저장 완료: {save_path}")

--- 시뮬레이션 설정 ---
총 시간: 10.0초
시간 간격(dt): 0.0005초 (0.5ms)
총 스텝 수: 20000 (메모리 주의)
시뮬레이션 시작...


W0000 00:00:1770359758.331236 3745487 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Progress: 0/20000 (0.0%)
Progress: 1000/20000 (5.0%)
Progress: 2000/20000 (10.0%)
Progress: 3000/20000 (15.0%)
Progress: 4000/20000 (20.0%)
Progress: 5000/20000 (25.0%)
Progress: 6000/20000 (30.0%)
Progress: 7000/20000 (35.0%)
Progress: 8000/20000 (40.0%)
Progress: 9000/20000 (45.0%)
Progress: 10000/20000 (50.0%)
Progress: 11000/20000 (55.0%)
Progress: 12000/20000 (60.0%)
Progress: 13000/20000 (65.0%)
Progress: 14000/20000 (70.0%)
Progress: 15000/20000 (75.0%)
Progress: 16000/20000 (80.0%)
Progress: 17000/20000 (85.0%)
Progress: 18000/20000 (90.0%)
Progress: 19000/20000 (95.0%)
데이터 변환 중... (시간이 소요될 수 있습니다)
최종 데이터 형태: (20000, 8, 64, 72, 2)
저장 완료: /data/hw/sionna/ws/output/bs1_ue7_channel_dataset_precise_10s.npy


In [ ]:

try:
    from sionna.phy.ofdm import ResourceGrid
except ImportError:
    from sionna.ofdm import ResourceGrid

# ==========================================
# 1. 파일 저장 경로 및 Scene 설정
# ==========================================
output_dir = "/data/hw/import os
import tensorflow as tf
import numpy as np
import sionna
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver

# [버전 호환성] Import 경로 처리sionna/ws/output"
os.makedirs(output_dir, exist_ok=True)

xml_path = "/data/hw/sionna/ws/scenes/kookmin/kookmin_itu.xml"
temp_xml_path = os.path.join(os.path.dirname(xml_path), "kookmin_fixed_absolute.xml")

# ==========================================
# 2. 시스템 및 시뮬레이션 파라미터
# ==========================================
carrier_frequency = 3.5e9
subcarrier_spacing = 30e3 
fft_size = 72
dt = 0.5e-3 
total_steps = 60000  # 10개 데이터만 수집

tx_positions = [[-125.663, 56.367, -181.453], [-50.472, 36.869, -181.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]
speeds_kmh = [30, 40, 50, 60, 70, 80, 90, 100]
speeds_ms = [v / 3.6 for v in speeds_kmh]
num_ues = len(speeds_kmh)

# ==========================================
# 3. 객체 및 경로 정의 (PolylineWalker)
# ==========================================
# road_positions는 이전 세션에서 로드된 상태여야 합니다.
path_indices = [
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
trajectory_points = road_positions[path_indices]

class PolylineWalker:
    def __init__(self, points):
        self.points = points
        diffs = points[1:] - points[:-1]
        self.seg_lengths = np.linalg.norm(diffs, axis=1)
        self.cum_dist = np.insert(np.cumsum(self.seg_lengths), 0, 0.0)
        
    def get_position(self, distance):
        total_len = self.cum_dist[-1]
        if distance >= total_len: return self.points[-1]
        if distance <= 0: return self.points[0]
        idx = np.searchsorted(self.cum_dist, distance) - 1
        ratio = (distance - self.cum_dist[idx]) / self.seg_lengths[idx] if self.seg_lengths[idx] > 0 else 0
        return self.points[idx] + (self.points[idx+1] - self.points[idx]) * ratio

walker = PolylineWalker(trajectory_points)

# ==========================================
# 4. Scene 구성
# ==========================================
scene = load_scene(temp_xml_path)
scene.tx_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

for name, pos in zip(tx_names, tx_positions):
    scene.add(Transmitter(name=name, position=pos))

ues = []
for i in range(num_ues):
    rx = Receiver(name=f"UE_{i}", position=trajectory_points[0])
    scene.add(rx)
    ues.append(rx)

## ==========================================
# 5. 시뮬레이션 루프 (에러 수정 지점)
# ==========================================
solver = PathSolver()
dataset_h = []

print("시뮬레이션 시작...")

for step in range(total_steps):
    current_time = step * dt
    for i, ue in enumerate(ues):
        ue.position = walker.get_position(speeds_ms[i] * current_time)
    
    paths = solver(scene, max_depth=3)
    frequencies = subcarrier_spacing * tf.range(fft_size, dtype=tf.float32)
    
    cfr_output = paths.cfr(frequencies=frequencies)
    
    # [1] 튜플/단일 텐서 여부 판별 후 Numpy 변환
    if isinstance(cfr_output, (list, tuple)):
        h_real = cfr_output[0].numpy()
        h_imag = cfr_output[1].numpy()
    else:
        h_np = cfr_output.numpy()
        h_real = h_np.real
        h_imag = h_np.imag

    # [2] 마지막 축에 Real/Imag 쌓기 (Complex -> 2)
    h_stacked = np.stack([h_real, h_imag], axis=-1)
    
    # [3] 에러 발생 지점 수정: 불필요한 차원(크기가 1인 차원)을 모두 자동으로 제거
    # 특정 axis를 지정하지 않고 np.squeeze(h_stacked)를 하면 크기가 1인 모든 차원이 사라집니다.
    h_squeezed = np.squeeze(h_stacked) 
    
    # [4] 차원 강제 조정 (예외 방지)
    # 최종 목표 구조: (UE, TX, BS_ANT, FREQ, 2) -> (6, 3, 64, 72, 2)
    # 만약 squeeze 후 차원이 꼬였다면 정확한 순서로 배치(reshape/transpose)가 필요할 수 있습니다.
    # 여기서는 데이터가 누락되지 않도록 그대로 담습니다.
    dataset_h.append(h_squeezed) 
    
    # 첫 스텝에서만 데이터 구조를 출력하여 확인 (디버깅용)
    if step == 0:
        print(f"Original Stacked Shape: {h_stacked.shape}")
        print(f"Squeezed Shape: {h_squeezed.shape}")
    if step % 1000 == 0:
        print(f"Step {step}/{total_steps} 완료")

# ==========================================
# 6. 저장
# ==========================================
dataset_h = np.array(dataset_h)
save_path = os.path.join(output_dir, "nlos_bs3_ue8_channel_dataset_precise_30s.npy")
np.save(save_path, dataset_h)

print("-" * 30)
print(f"저장 완료. 최종 데이터 형태: {dataset_h.shape}")

시뮬레이션 시작...
Original Stacked Shape: (8, 1, 3, 64, 1, 72, 2)
Squeezed Shape: (8, 3, 64, 72, 2)
Step 0/20000 완료
Step 1000/20000 완료
Step 2000/20000 완료
Step 3000/20000 완료
Step 4000/20000 완료
Step 5000/20000 완료
Step 6000/20000 완료
Step 7000/20000 완료
Step 8000/20000 완료
Step 9000/20000 완료
Step 10000/20000 완료
Step 11000/20000 완료
Step 12000/20000 완료
Step 13000/20000 완료
Step 14000/20000 완료
Step 15000/20000 완료
Step 16000/20000 완료
Step 17000/20000 완료
Step 18000/20000 완료
Step 19000/20000 완료
------------------------------
저장 완료. 최종 데이터 형태: (20000, 8, 3, 64, 72, 2)
